In [1]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

#pl.Config.set_tbl_rows(-1)             # shows all table rows globably
#pl.Config.set_tbl_cols(-1)             # shows all table columns globably
#pl.Config.set_fmt_str_lengths(100)     # increase character count visible globably
#pl.Config.restore_defaults()           # reset to default configuration

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_CET_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data).with_columns(pl.col(["Price_profile", "Price_level", "Region", "Which_Phases_Participate",
                                                                "Municipality", "Group_Phase2", "Control_Price_Phase2"]).cast(pl.Categorical))

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_CET_data.parquet


In [2]:
# Check for missing values
def what_percentage_missing(df):
    missing_percentage = (
        df.select((pl.all().is_null().sum() / pl.len()) * 100)
        .transpose(include_header=True, header_name="column", column_names=["pct_missing"])
        .sort("pct_missing", descending=True)
    )

    print("Percentage of missing values in each column:")

    with pl.Config(tbl_rows=-1):
        print(missing_percentage)

what_percentage_missing(df)

Percentage of missing values in each column:
shape: (19, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 88.006897   │
│ Price_level              ┆ 88.006897   │
│ Experiment_price_NOK_kWh ┆ 88.006897   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [3]:
# Create a boolean expression checking for "Control"
is_control = pl.col("Group_Phase2").cast(pl.String).str.contains("Control")

# Summary DataFrame with counts and percentages
summary_df = (
    df.with_columns(
        pl.when(is_control)
        .then(pl.lit("Control"))
        .otherwise(pl.lit("Experiment"))
        .alias("Cohort")
    )
    .group_by("Cohort")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
    )
)

print(summary_df)

shape: (2, 3)
┌────────────┬─────────┬───────┐
│ Cohort     ┆ count   ┆ pct   │
│ ---        ┆ ---     ┆ ---   │
│ str        ┆ u32     ┆ f64   │
╞════════════╪═════════╪═══════╡
│ Experiment ┆ 8638752 ┆ 70.93 │
│ Control    ┆ 3541248 ┆ 29.07 │
└────────────┴─────────┴───────┘


In [4]:
# Subset containing Ber_Control, Bo_Control, Control, Os_Control, Trom_Control, Trond_Control
df_control = df.filter(is_control)

# Subset containing everything else (Ber_1, Bo_1, H1, H2, NA, Os_1..6, Trom_1..2, Trond_1)
df_experiment = df.filter(~is_control)

In [5]:
# just the dataframe with 
what_percentage_missing(df_experiment)

Percentage of missing values in each column:
shape: (19, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 83.090613   │
│ Price_level              ┆ 83.090613   │
│ Experiment_price_NOK_kWh ┆ 83.090613   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [6]:
# 100% missing - as expected
#what_percentage_missing(df_control)

In [7]:
# How many and what proportion of all days fall specifically on weekends: 2,376,394 Control-hours plus another 9,813 Experiment-hours are from a Weekend date 

total_rows = df.height
total_cols = df.width

stats = (
    df
    .filter(
        (pl.col("Is_Weekend") == 1)
        # & (pl.col("Experiment_price_NOK_kWh").is_not_null())
        # & (pl.col("Price_profile").is_not_null())
        # & (pl.col("Price_level").is_not_null())
    )
    .select(
        Match_Count=pl.len(),
        Control_Null_Count=pl.col("Price_level").is_null().sum(),
    )
    .with_columns(
        Pct_of_Total_DF=(pl.col("Match_Count") / total_rows) * 100,
        Control_Null_Pct_of_Total_DF=(pl.col("Control_Null_Count") / total_rows) * 100,
        Control_Null_Pct_of_Matches=(pl.col("Control_Null_Count") / pl.col("Match_Count")) * 100,
    )
)

print(f"Total rows in df: {total_rows} and Total columns in df: {total_cols}")
print(stats)

Total rows in df: 12180000 and Total columns in df: 19
shape: (1, 5)
┌─────────────┬────────────────────┬─────────────────┬──────────────────────┬──────────────────────┐
│ Match_Count ┆ Control_Null_Count ┆ Pct_of_Total_DF ┆ Control_Null_Pct_of_ ┆ Control_Null_Pct_of_ │
│ ---         ┆ ---                ┆ ---             ┆ Total_DF             ┆ Matches              │
│ u32         ┆ u32                ┆ f64             ┆ ---                  ┆ ---                  │
│             ┆                    ┆                 ┆ f64                  ┆ f64                  │
╞═════════════╪════════════════════╪═════════════════╪══════════════════════╪══════════════════════╡
│ 3364375     ┆ 3354562            ┆ 27.622126       ┆ 27.54156             ┆ 99.708326            │
└─────────────┴────────────────────┴─────────────────┴──────────────────────┴──────────────────────┘


In [8]:
# Very few occurances of Expermiental days - it is possible that some of these are just "bleed" from a Friday night midnight into Saturday
# We can not create an Is_Weekend variant of our proposed model, with these data. Bummer.
weekend_anomalies = (
    df
    .filter((pl.col("Is_Weekend") == 1) & (pl.col("Price_level").is_not_null()))
    .group_by(["Municipality", "Price_profile", "Hour"])
    .len()
    .sort("len", descending=True)
)

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(weekend_anomalies)

# Optional: Remove the 3,364,375 occurances of Control Weekend + 9,813 experiment occurances bleed into the weekend - so that the Is_Weekend column can be dropped
# Also this will ensure that our Baseline kWh measurement only uses weekday values from control - as those are the only days the Experiment was conducted
df_no_weekends = df.remove(pl.col("Is_Weekend") == 1)
print(f"Total rows in df: {df_no_weekends.height} and Total columns in df: {df_no_weekends.width}")


shape: (33, 4)
┌──────────────┬───────────────┬──────┬──────┐
│ Municipality ┆ Price_profile ┆ Hour ┆ len  │
│ ---          ┆ ---           ┆ ---  ┆ ---  │
│ cat          ┆ cat           ┆ i32  ┆ u32  │
╞══════════════╪═══════════════╪══════╪══════╡
│ Oslo         ┆ P             ┆ 0    ┆ 1076 │
│ Tromsø       ┆ P             ┆ 0    ┆ 1060 │
│ Tromsø       ┆ B             ┆ 0    ┆ 1036 │
│ Bergen       ┆ B             ┆ 0    ┆ 807  │
│ Trondheim    ┆ B             ┆ 0    ┆ 804  │
│ Oslo         ┆ B             ┆ 0    ┆ 648  │
│ Oslo         ┆ A             ┆ 0    ┆ 454  │
│ Oslo         ┆ C             ┆ 0    ┆ 449  │
│ Bodø         ┆ B             ┆ 0    ┆ 442  │
│ Oslo         ┆ P0            ┆ 0    ┆ 436  │
│ Bærum        ┆ P             ┆ 0    ┆ 337  │
│ Bærum        ┆ B             ┆ 0    ┆ 192  │
│ Lillestrøm   ┆ P             ┆ 0    ┆ 161  │
│ Stavanger    ┆ P             ┆ 0    ┆ 156  │
│ Stavanger    ┆ B             ┆ 0    ┆ 156  │
│ Bergen       ┆ P             ┆ 0    ┆ 138  

In [9]:
# Check for duplicate rows
duplicate_rows = df.is_duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [10]:
# Check data types
print("Data types of each column:")
schema_df = pl.DataFrame({
    "Column": df_no_weekends.columns,
    "Data Type": [str(dtype) for dtype in df.dtypes]})

with pl.Config(tbl_rows=-1):
    print(schema_df)

Data types of each column:
shape: (19, 2)
┌──────────────────────────┬─────────────────────────────────┐
│ Column                   ┆ Data Type                       │
│ ---                      ┆ ---                             │
│ str                      ┆ str                             │
╞══════════════════════════╪═════════════════════════════════╡
│ ID                       ┆ String                          │
│ From                     ┆ Datetime(time_unit='ms', time_… │
│ Date                     ┆ Date                            │
│ Day_Of_Week              ┆ Int32                           │
│ Is_Weekend               ┆ Int32                           │
│ Hour                     ┆ Int32                           │
│ Demand_kWh               ┆ Float64                         │
│ Price_profile            ┆ Categorical                     │
│ Price_level              ┆ Categorical                     │
│ Experiment_price_NOK_kWh ┆ Float64                         │
│ Temperature

In [ ]:
# Calculate the target label (Y)


In [11]:
# Audit and filter raw records following Hofmann & Lindberg's quality protocol:
# - Remove negative values, extreme spikes (>25 kWh), and anomalous households (mean > 9.9 kWh).
# - Identify and flag multi-day consecutive zero-kWh runs (vacant homes / meter dropouts).

print("Data Quality Audit")
#initial_row_count = df.height
initial_row_count = df_no_weekends.height
print(f"Starting with {initial_row_count} rows")
initial_households = df.select("ID").n_unique()

# Filter out invalid individual hourly readings (negative values, extreme spikes > 25 kWh)
df_audited = df_no_weekends.filter(
    (pl.col("Demand_kWh") >= 0.0) & 
    (pl.col("Demand_kWh") <= 25.0)
)
rows_after_hourly_filter = df_audited.height
print(f"Removed {initial_row_count - rows_after_hourly_filter} invalid hourly readings (negative or >25 kWh).")

# Exclude households whose overall mean consumption exceeds 9.9 kWh
household_means = (
    df_audited
    .group_by("ID")
    .agg(pl.col("Demand_kWh").mean().alias("Mean_Demand"))
)

valid_households = (
    household_means
    .filter(pl.col("Mean_Demand") <= 9.9)
    .select("ID")
)

df_audited = df_audited.join(valid_households, on="ID", how="inner")
rows_after_household_filter = df_audited.height
excluded_households = initial_households - df_audited.select("ID").n_unique()
print(f"Excluded {excluded_households} households with mean consumption > 9.9 kWh.")
print(f"Rows remaining after household-level filter: {rows_after_household_filter}")


print("Running Robust Consecutive Zero Audit")

# Ensure data is strictly sorted by ID and chronological Timestamp/Date-Hour
df_audited = df_audited.sort(["ID", "Date", "Hour"])

# Flag rows where demand is exactly zero
df_audited = df_audited.with_columns(
    Is_Zero=(pl.col("Demand_kWh") == 0.0)
)

print("test of audited")
#print(df_audited.head())
print(df_audited.select(pl.col("Is_Zero").not_().sum().alias("false_count")))
#print(df_audited)
# Only 668 instances found where Demand is 0 kWh 0.006% of total rows - we will keep these in the dataset
# Optional removal
#df_audited = df_audited.remove(pl.col("Is_Zero") == True)

# View shape of dataframe
print(f"Total rows in df: {df_audited.height} and Total columns in df: {df_audited.width}")

Data Quality Audit
Starting with 8815625 rows
Removed 0 invalid hourly readings (negative or >25 kWh).
Excluded 0 households with mean consumption > 9.9 kWh.
Rows remaining after household-level filter: 8815625
Running Robust Consecutive Zero Audit
test of audited
shape: (1, 1)
┌─────────────┐
│ false_count │
│ ---         │
│ u32         │
╞═════════════╡
│ 8814957     │
└─────────────┘
Total rows in df: 8815625 and Total columns in df: 20


In [14]:
epsilon = 1e-6  # Smoothing constant to prevent division-by-zero

# AGGREGATE HOURLY OBSERVATIONS TO HOUSEHOLD-DAY LEVEL (KEEP ALL COLUMNS)
# Note: Adjust non-aggregated columns if any extra columns were added to df_audited
meta_cols = [
    c for c in df_audited.columns
    if c not in {
        "ID", "Date", "Hour", "From", "Demand_kWh",
        "Experiment_price_NOK_kWh", "Temperature",
        "Temperature24", "Temperature48", "Temperature72",
        "Temperature_Bin", "Is_Zero"
    }
]

df_daily = (
    df_audited
    .sort(["ID", "Date", "Hour"])
    .group_by(["ID", "Date"], maintain_order=True)
    .agg(
        [
            pl.col("From").first().alias("From"),
            pl.col("Demand_kWh").sum().alias("E_actual_peak"),
            pl.col("Experiment_price_NOK_kWh").sum().alias("Experiment_price_NOK_kWh"),
            pl.col("Temperature").mean().alias("Temperature"),
            pl.col("Temperature24").mean().alias("Temperature24"),
            pl.col("Temperature48").mean().alias("Temperature48"),
            pl.col("Temperature72").mean().alias("Temperature72"),
            # Count how many hours on this date had an active experiment price
            pl.col("Experiment_price_NOK_kWh").is_not_null().sum().alias("Active_Exp_Hours"),
        ]
        + [pl.col(c).first().alias(c) for c in meta_cols]
    )
    .with_columns(
        # Recalculate 5°C temperature bin from daily mean Temperature
        Temperature_Bin=((pl.col("Temperature") / 5.0).round() * 5.0).alias("Temperature_Bin")
    )
    .sort(["ID", "Date"])
)

# ==============================================================================
# 2. CALCULATE E_base_peak FROM UP TO 10 MOST RECENT PRIOR DAYS (NO LEAKAGE)
# ==============================================================================
# Shift by 1 day within each household so day `d` only uses prior days (< d).
# Using rolling_mean over a 10-row window after shift(1) gives the mean of up to
# 10 immediately preceding observations for that household.
df_daily_baseline = (
    df_daily
    .sort(["ID", "Date"])
    .with_columns(
        E_base_peak=(
            pl.col("E_actual_peak")
            .shift(1)
            .rolling_mean(window_size=10, min_samples=1)
            .over("ID")
        ),
        Prior_10d_Mean_Temp=(
            pl.col("Temperature")
            .shift(1)
            .rolling_mean(window_size=10, min_samples=1)
            .over("ID")
        ),
        Prior_10d_Obs_Count=(
            pl.col("E_actual_peak")
            .shift(1)
            .is_not_null()
            .cast(pl.Int32)
            .rolling_sum(window_size=10, min_samples=1)
            .over("ID")
            .fill_null(0)
        ),
    )
)

# ==============================================================================
# 3. FILTER TO HOUSEHOLD-EXPERIMENT-DAYS & COMPUTE FPI TARGETS
# ==============================================================================
# Household-experiment-days: days with active experiment pricing (Active_Exp_Hours > 0)
# and at least 1 prior day available for E_base_peak.
df_fpi_experiment = (
    df_daily_baseline
    .filter(
        (pl.col("Active_Exp_Hours") > 0) &
        (pl.col("E_base_peak").is_not_null())
    )
    .with_columns(
        Flexibility_Performance_Index=(
            (pl.col("E_base_peak") - pl.col("E_actual_peak")) /
            (pl.col("E_base_peak") + epsilon)
        )
    )
    .with_columns(
        High_Flexibility_Flag=(pl.col("Flexibility_Performance_Index") > 0.05).cast(pl.Int32)
    )
)

print(f"Daily rows (all): {df_daily_baseline.height:,} | Columns: {df_daily_baseline.width}")
print(f"Household-experiment-day rows (with FPI): {df_fpi_experiment.height:,} | Columns: {df_fpi_experiment.width}")
print(
    df_fpi_experiment.select([
        "ID", "Date", "E_base_peak", "E_actual_peak",
        "Temperature", "Temperature_Bin",
        "Flexibility_Performance_Index", "High_Flexibility_Flag"
    ]).head(10)
)

Daily rows (all): 367,500 | Columns: 23
Household-experiment-day rows (with FPI): 95,982 | Columns: 25
shape: (10, 8)
┌───────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ ID    ┆ Date       ┆ E_base_pea ┆ E_actual_p ┆ Temperatur ┆ Temperatur ┆ Flexibilit ┆ High_Flexi │
│ ---   ┆ ---        ┆ k          ┆ eak        ┆ e          ┆ e_Bin      ┆ y_Performa ┆ bility_Fla │
│ str   ┆ date       ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ nce_Index  ┆ g          │
│       ┆            ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ ---        ┆ ---        │
│       ┆            ┆            ┆            ┆            ┆            ┆ f64        ┆ i32        │
╞═══════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╡
│ Exp_1 ┆ 2020-12-16 ┆ 58.6255    ┆ 55.653     ┆ 7.733333   ┆ 10.0       ┆ 0.050703   ┆ 1          │
│ Exp_1 ┆ 2020-12-17 ┆ 58.4279    ┆ 57.579     ┆ 8.279167   ┆ 10.0       ┆